In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyignite import Client
from datetime import datetime
import time

# Simulate sensor data (in real-world, this would come from actual sensors)
def generate_sensor_data(num_records=100):
    """Generate synthetic rainfall and river level data."""
    timestamps = [datetime.now() for _ in range(num_records)]
    rainfall = np.random.uniform(0, 50, num_records)  # mm/hour
    river_level = np.random.uniform(0.5, 5.0, num_records)  # meters
    return pd.DataFrame({
        'timestamp': timestamps,
        'rainfall': rainfall,
        'river_level': river_level
    })

# Traditional computing: Preprocess data
def preprocess_data(df):
    """Clean and normalize sensor data."""
    df = df.dropna()  # Remove missing values
    # Normalize data for consistency
    df['rainfall_norm'] = (df['rainfall'] - df['rainfall'].min()) / (df['rainfall'].max() - df['rainfall'].min())
    df['river_level_norm'] = (df['river_level'] - df['river_level'].min()) / (df['river_level'].max() - df['river_level'].min())
    return df

# In-memory computing: Process data with Apache Ignite
def compute_flood_risk_ignite(df):
    """Use Ignite to compute flood risk scores in memory."""
    # Connect to Ignite cluster (assumes Ignite node running locally)
    client = Client()
    client.connect('127.0.0.1', 10800)  # Default Ignite port

    # Create or get a cache for sensor data
    cache = client.get_or_create_cache('sensor_data')

    # Store preprocessed data in Ignite cache
    for idx, row in df.iterrows():
        cache.put(idx, {
            'timestamp': str(row['timestamp']),
            'rainfall_norm': row['rainfall_norm'],
            'river_level_norm': row['river_level_norm']
        })

    # Compute flood risk in memory (simple weighted sum for demo)
    # In practice, use hydrological models or ML
    risk_scores = []
    for idx in range(len(df)):
        data = cache.get(idx)
        # Example: Flood risk = 0.6 * rainfall_norm + 0.4 * river_level_norm
        risk = 0.6 * data['rainfall_norm'] + 0.4 * data['river_level_norm']
        risk_scores.append(risk)

    # Clean up
    client.close()
    return np.array(risk_scores)

# Traditional computing: Visualize results
def visualize_results(df, risk_scores):
    """Plot sensor data and flood risk."""
    plt.figure(figsize=(10, 6))
    plt.plot(df['timestamp'], df['rainfall'], label='Rainfall (mm/h)', color='blue')
    plt.plot(df['timestamp'], df['river_level'], label='River Level (m)', color='green')
    plt.plot(df['timestamp'], risk_scores, label='Flood Risk Score', color='red', linestyle='--')
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.title('Flood Risk Prediction')
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Main execution
def main():
    # Step 1: Generate synthetic sensor data (traditional computing)
    print("Generating sensor data...")
    sensor_data = generate_sensor_data()

    # Step 2: Preprocess data (traditional computing)
    print("Preprocessing data...")
    processed_data = preprocess_data(sensor_data)

    # Step 3: Compute flood risk using in-memory computing
    print("Computing flood risk with Apache Ignite...")
    start_time = time.time()
    risk_scores = compute_flood_risk_ignite(processed_data)
    print(f"IMC processing time: {time.time() - start_time:.2f} seconds")

    # Step 4: Visualize results (traditional computing)
    print("Visualizing results...")
    visualize_results(processed_data, risk_scores)

if __name__ == "__main__":
    main()

Failed to perform handshake, connection to node(address=127.0.0.1, port=10800) with protocol context ProtocolContext(version=(1, 7, 0), features=4) failed: [Errno 61] Connection refused
Traceback (most recent call last):
  File "/Users/bkyanjo3/firedrake/lib/python3.10/site-packages/pyignite/connection/connection.py", line 238, in connect
    result = self._connect_version()
  File "/Users/bkyanjo3/firedrake/lib/python3.10/site-packages/pyignite/connection/connection.py", line 268, in _connect_version
    self._socket.connect((self.host, self.port))
ConnectionRefusedError: [Errno 61] Connection refused


Generating sensor data...
Preprocessing data...
Computing flood risk with Apache Ignite...


ReconnectError: Can not connect.

In [2]:
pip install pyignite 

  Preparing metadata (setup.py) ... done
  Created wheel for pyignite: filename=pyignite-0.6.1-cp310-cp310-macosx_14_0_arm64.whl size=134558 sha256=0460eb2b68ccf9b41f7f0da2f7d1590cf16f0ada9e4ac9fc30a71d9b77ae1876
  Stored in directory: /Users/bkyanjo3/Library/Caches/pip/wheels/46/b1/4f/856dea00e90ee82f608709937eb37612447602804de590e894
Successfully built pyignite
Note: you may need to restart the kernel to use updated packages.
